# Gap to the Podium: Phase 3 analysis

How far is India from an Olympic medal in each event, how sure can we be, and is the gap closing?
Data: BigQuery marts and staging views built in Phase 2 (Olympedia, Wikipedia, World Athletics).

In [1]:
# Make the project's src/ package importable when the notebook runs from notebooks/.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))


from src.warehouse import table

gap = table("marts.mart_gap_to_podium")
toplists = table("staging.stg_wa_toplists")
gap.shape, toplists.shape

((42, 18), (20934, 18))

## 1. A fair threshold: medallists' season bests

Olympic distance finals are often slow and tactical, so their bronze marks understate medal level.
For each of the last three Games, each podium finisher is matched to their season best on the
World Athletics toplist of that season, and the same bronze rule (third-best on the podium) is
applied to season bests.

In [2]:
from src.analysis import average_over_games, gap_pct, match_medallists, season_best_bronze
from src.warehouse import query

podium = query("""
    SELECT championship_year, event_key, athlete_name, country_code, position, mark_value,
           higher_is_better, discipline_group
    FROM staging.stg_olympic_results
    WHERE position BETWEEN 1 AND 3 AND championship_year IN (2016, 2020, 2024)
""")
matched = match_medallists(podium, toplists)
matched["match"].value_counts()

match
name+country       356
surname+country     15
not found           11
name                 2
ambiguous            1
Name: count, dtype: int64

In [3]:
unmatched = matched[~matched["match"].isin(["name+country", "name"])]
unmatched[["championship_year", "event_key", "athlete_name", "country_code", "match"]]

,championship_year,event_key,athlete_name,country_code,match
23,2016,marathon_m,Galen Rupp,USA,not found
28,2016,400mh_m,Boniface Tumuti,KEN,not found
36,2016,50kmw_m,Matej Tóth,SVK,not found
37,2016,50kmw_m,Jared Tallent,AUS,not found
38,2016,50kmw_m,Hirooki Arai,JPN,not found
58,2016,ht_m,Ivan Tsikhan,BLR,not found
101,2016,20kmw_w,Lü Xiuzhi,CHN,surname+country
126,2016,hep_w,Nafi Thiam,BEL,surname+country
133,2020,200m_m,Kenny Bednarek,USA,surname+country
145,2020,5000m_m,Moh Ahmed,CAN,surname+country


In [4]:
sb = average_over_games(season_best_bronze(matched))
fair = gap.merge(sb, on="event_key", how="left")
fair["gap_sb_pct"] = gap_pct(fair["india_best_mark"], fair["sb_threshold"],
                             fair["higher_is_better"]).round(2)
cols = ["event_key", "india_best_athlete", "india_best_mark_raw", "gap_pct", "gap_sb_pct",
        "india_world_rank", "tactical_event", "games_used"]
fair.sort_values("gap_sb_pct")[cols]

,event_key,india_best_athlete,india_best_mark_raw,gap_pct,gap_sb_pct,india_world_rank,tactical_event,games_used
8,jt_m,Neeraj CHOPRA,90.23,-4.37,-2.86,3,False,3
12,10000m_m,Gulveer SINGH,27:00.22,-0.67,-0.66,12,True,3
18,lj_m,SREESHANKAR,8.38,-1.21,-0.24,9,False,3
36,5000m_m,Gulveer SINGH,13:03.93,-0.29,-0.20,38,True,3
5,tj_m,Praveen CHITHRAVEL,17.37,1.10,1.29,10,False,3
35,3000msc_w,Parul CHAUDHARY,9:08.67,1.10,1.34,11,True,3
17,lj_w,Ancy SOJAN EDAPPILLY,6.88,1.76,2.09,12,False,3
16,hj_m,Sarvesh Anil KUSHARE,2.31,1.56,2.12,6,False,3
9,800m_m,Mohammed Afsal PULIKKALAKATH,1:44.93,1.60,2.16,101,True,3
19,100m_m,Gurindervir SINGH,10.09,2.23,2.23,106,False,3


In [5]:
print(matched["match"].value_counts(), "\n")
print("inconsistent matches (left out):")
print(matched[matched["sb_consistent"].eq(False)][["championship_year", "event_key", "athlete_name",
                                                    "mark_value", "season_best"]])
matched[matched["match"] == "surname+country"][["championship_year", "event_key", "athlete_name",
                                                 "mark_value", "season_best", "sb_consistent"]]

match
name+country       356
surname+country     15
not found           11
name                 2
ambiguous            1
Name: count, dtype: int64 

inconsistent matches (left out):
Empty DataFrame
Columns: [championship_year, event_key, athlete_name, mark_value, season_best]
Index: []


,championship_year,event_key,athlete_name,mark_value,season_best,sb_consistent
101,2016,20kmw_w,Lü Xiuzhi,5322.00,5287.00,True
126,2016,hep_w,Nafi Thiam,6810.00,6810.00,True
133,2020,200m_m,Kenny Bednarek,19.68,19.68,True
145,2020,5000m_m,Moh Ahmed,778.61,770.12,True
174,2020,pv_m,Mondo Duplantis,6.02,6.10,True
175,2020,pv_m,Chris Nilsen,5.97,5.97,True
220,2020,100mh_w,Keni Harrison,12.52,12.47,True
255,2020,hep_w,Nafi Thiam,6791.00,6791.00,True
262,2024,200m_m,Kenny Bednarek,19.62,19.57,True
265,2024,400m_m,Matt Hudson-Smith,43.44,43.44,True


In [6]:
check = matched[matched["event_key"].isin(["5000m_m", "10000m_m"])]
check[["event_key", "championship_year", "athlete_name", "mark_value", "season_best",
       "world_rank", "match"]].sort_values(["event_key", "championship_year", "season_best"])

,event_key,championship_year,athlete_name,mark_value,season_best,world_rank,match
18,10000m_m,2016,Mo Farah,1625.17,1613.71,2.0,name+country
20,10000m_m,2016,Tamirat Tola,1626.26,1617.33,4.0,name+country
19,10000m_m,2016,Paul Tanui,1625.64,1625.64,10.0,name+country
149,10000m_m,2020,Jacob Kiplimo,1663.88,1593.93,1.0,name+country
147,10000m_m,2020,Selemon Barega,1663.22,1609.51,2.0,name+country
148,10000m_m,2020,Joshua Cheptegei,1663.63,1663.63,62.0,name+country
277,10000m_m,2024,Berihu Aregawi,1603.44,1591.13,2.0,name+country
276,10000m_m,2024,Joshua Cheptegei,1603.14,1603.14,6.0,name+country
278,10000m_m,2024,Grant Fisher,1603.46,1603.46,7.0,name+country
15,5000m_m,2016,Mo Farah,783.30,779.29,1.0,name+country


## 2. Hypothesis: season-best speed predicts medals less well in tactical events

H1: medallists in tactical events (middle distance, long distance, road) have worse pre-Games
season-best world ranks than medallists in all-out events. One-sided Mann-Whitney U test,
medallists at the 2016, 2021 and 2024 Olympics.

In [7]:
import pandas as pd

from src.analysis import medallist_rank_test

pd.Series(medallist_rank_test(matched)).round(4)

n_tactical                        124.0000
n_all_out                         249.0000
median_rank_tactical                4.0000
median_rank_all_out                 3.0000
share_outside_top10_tactical        0.2258
share_outside_top10_all_out         0.0321
u_statistic                     19807.5000
p_value                             0.0000
dtype: float64

**Result.** Supported (one-sided Mann-Whitney U, p < 0.0001; 124 tactical and 249 all-out
medallists, 2016-2024). Median season-best world ranks are similar (4 vs 3), but 22.6% of
tactical-event medallists were ranked outside the world top 10 that season, against 3.2% in
all-out events. In all-out events a medallist is almost always a top-10 performer; in tactical
events about one in four is not. Caveat: some athletes medalled more than once, so observations
are not fully independent.

**Implication.** No time threshold is fair for tactical events (finals are slow, and medallists'
season bests are often modest). World rank is the better measure there, and a rank just outside
the top 10 (Parul Chaudhary, Gulveer Singh) is a more realistic medal position than the same
rank in an all-out event.